## 🎯 Learning Objectives
* Understand the concept and importance of CI-integrated evaluations for LLM-powered applications.
* Learn how to integrate automated LLM evaluation into Continuous Integration (CI) pipelines.
* Identify key metrics and tools for evaluating LLM performance in a CI/CD context.
* Analyze the trade-offs and best practices for implementing CI-integrated LLM evaluations.


## CI-integrated Evals: Running Tests on Every Pull Request

In the world of traditional software development, Continuous Integration (CI) is a cornerstone practice. Every time a developer pushes code, automated tests (unit, integration, end-to-end) run to catch regressions and ensure code quality. For Large Language Model (LLM) applications, this concept is not just relevant but absolutely critical. Welcome to **CI-integrated evaluations**, where we extend the safety net of CI to the nuanced and often unpredictable behavior of LLMs.

### The Analogy: Unit Tests for LLM Behavior

Imagine you're building a complex web application. Every new feature or bug fix could inadvertently break existing functionality. To prevent this, you write unit tests that assert specific parts of your code behave as expected. If a test fails, the Pull Request (PR) is blocked, preventing faulty code from reaching production.

For LLM applications, the 'code' isn't just Python or JavaScript; it's also the prompt engineering, the retrieval strategy in a RAG system, the fine-tuned model weights, and the orchestration logic. A small change in a prompt, an update to your vector database, or a new version of an underlying LLM could subtly (or drastically) alter your application's responses. CI-integrated evaluations are our 'unit tests' for LLM behavior. They automatically assess the quality, safety, and performance of your LLM application on every PR, ensuring that changes don't introduce regressions or undesirable behaviors.

### Why is it Crucial for LLMOps?

1.  **Early Regression Detection:** Catch issues like hallucination, irrelevant responses, or safety violations *before* they merge into your main branch. This saves significant debugging time and prevents negative user experiences.
2.  **Maintain Model Quality:** Ensure that updates to your RAG corpus, prompt templates, or even the LLM itself don't degrade the quality of responses for critical use cases.
3.  **Consistency and Reliability:** Guarantee a consistent level of performance and adherence to guidelines across different development cycles and team members.
4.  **Accelerated Iteration:** Developers can iterate faster with confidence, knowing that an automated guardian is checking their changes.
5.  **Compliance and Safety:** For sensitive applications, automated evaluations can help enforce safety guardrails, detect bias, and ensure compliance with regulatory requirements.

### How it Works (The Modern 2026 Stack)

In 2026, CI-integrated evaluations for LLMs typically involve:

*   **CI/CD Platform:** Cloud-native solutions like GitHub Actions, GitLab CI, Azure DevOps Pipelines, or Google Cloud Build are standard. These platforms orchestrate the entire process.
*   **Evaluation Frameworks:** Specialized LLM evaluation libraries like `ragas`, `deepeval`, or `truera` are used to define metrics (e.g., faithfulness, answer relevance, context recall, toxicity, bias) and run tests. These often integrate with LLM providers or local inference engines.
*   **Test Datasets:** A curated set of questions, contexts, and expected answers (ground truth) that represent critical use cases for your application. These datasets are version-controlled alongside your code.
*   **LLM Inference:** The CI pipeline will invoke your LLM application (e.g., a RAG pipeline, a chatbot agent) with the test dataset.
*   **Metric Calculation & Reporting:** The evaluation framework calculates scores based on the LLM's responses against the ground truth and predefined criteria. Results are then reported back to the CI platform, often with thresholds that determine PR pass/fail status.
*   **Observability Integration:** Results are pushed to an observability platform (e.g., Weights & Biases, MLflow, custom dashboards) for historical tracking and deeper analysis.

This lesson will walk you through a practical example of setting up a simplified CI-integrated evaluation using Python, focusing on a RAG application and leveraging modern evaluation tools.


In [ ]:
# Install necessary libraries (run this once in your environment)
# pip install langchain-core langchain-community ragas datasets

import os
from typing import List, Dict, Any
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevance, context_recall

# --- 1. Simulate a RAG Application --- 
# In a real scenario, this would involve calling an actual LLM and a retrieval system.
# For this CI simulation, we'll use a deterministic mock.

class MockRAGSystem:
    def __init__(self):
        # Simulate a knowledge base
        self.knowledge_base = {
            "What is AgenticLabs.ng?": {
                "context": "AgenticLabs.ng is a leading AI research and development lab focused on building advanced agentic AI systems and automation tools. They specialize in LLMOps, autonomous agents, and AI-driven workflow optimization.",
                "response": "AgenticLabs.ng is an AI research and development lab specializing in agentic AI, LLMOps, and AI-driven automation."
            },
            "What is LLMOps?": {
                "context": "LLMOps (Large Language Model Operations) is a set of practices for managing the lifecycle of Large Language Models, from development and experimentation to deployment, monitoring, and continuous evaluation in production environments.",
                "response": "LLMOps refers to the practices for managing the entire lifecycle of Large Language Models, including development, deployment, monitoring, and continuous evaluation."
            },
            "Who founded AgenticLabs.ng?": {
                "context": "AgenticLabs.ng was founded by a team of experienced AI researchers and engineers with a vision to democratize advanced AI capabilities. Specific founder names are not publicly disclosed in this simulated knowledge base.",
                "response": "The founders of AgenticLabs.ng are experienced AI researchers and engineers, though specific names are not publicly disclosed."
            }
        }

    def query(self, question: str) -> Dict[str, str]:
        """Simulates querying the RAG system."""
        if question in self.knowledge_base:
            return {
                "answer": self.knowledge_base[question]["response"],
                "contexts": [self.knowledge_base[question]["context"]]
            }
        else:
            return {
                "answer": "I'm sorry, I don't have information on that topic in my knowledge base.",
                "contexts": []
            }

# --- 2. Define Test Cases (Evaluation Dataset) ---
# In a real CI pipeline, this dataset would be version-controlled.

eval_examples = [
    {
        "question": "What is AgenticLabs.ng?",
        "ground_truth": "AgenticLabs.ng is an AI research and development lab focused on agentic AI and automation."
    },
    {
        "question": "What does LLMOps stand for?",
        "ground_truth": "LLMOps stands for Large Language Model Operations, covering the lifecycle management of LLMs."
    },
    {
        "question": "Who are the founders of AgenticLabs.ng?",
        "ground_truth": "The founders are experienced AI researchers and engineers, but their names are not publicly disclosed."
    },
    {
        "question": "Tell me about the history of the internet.", # Out-of-scope question
        "ground_truth": "This question is outside the scope of AgenticLabs.ng's knowledge base."
    }
]

# --- 3. Run the RAG System against Test Cases and Collect Responses ---

rag_system = MockRAGSystem()

# Prepare data for Ragas evaluation
ragas_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

print("Running RAG system against test cases...")
for example in eval_examples:
    question = example["question"]
    ground_truth = example["ground_truth"]
    
    rag_output = rag_system.query(question)
    
    ragas_data["question"].append(question)
    ragas_data["answer"].append(rag_output["answer"])
    ragas_data["contexts"].append(rag_output["contexts"])
    ragas_data["ground_truth"].append(ground_truth)
    
    print(f"  Q: {question}\n  A: {rag_output['answer']}\n  C: {rag_output['contexts']}\n")

# Create a Ragas Dataset object
dataset = Dataset.from_dict(ragas_data)

# --- 4. Perform Evaluation using Ragas ---
# In a real CI, you might configure an actual LLM for Ragas to use for evaluation
# For local execution without API keys, Ragas can sometimes use local models or mock LLMs.
# Here, we'll assume Ragas has access to an LLM for its internal metric calculations.
# For demonstration, we'll use a simplified setup that doesn't require an actual LLM API key
# if the metrics can be calculated based on provided answers/contexts (some can, some need LLM).
# For faithfulness and answer_relevance, Ragas typically uses an LLM. 
# For this example, we'll simulate the evaluation step.

print("\nStarting Ragas evaluation...")

# NOTE: For a full Ragas evaluation, you would typically configure an LLM provider.
# Example: os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
# from langchain_openai import ChatOpenAI
# llm = ChatOpenAI(model="gpt-4o")
# embeddings = OpenAIEmbeddings()

# For this simulation, we'll proceed with the evaluation call, 
# assuming necessary LLM/embedding configurations are handled externally or mocked.
# In a real CI, you'd ensure these are properly set up.

# Define the metrics to evaluate
metrics = [
    faithfulness,
    answer_relevance,
    context_recall
]

# Evaluate the dataset
# If you have an LLM configured (e.g., via environment variables for OpenAI),
# Ragas will use it. Otherwise, it might raise an error or use a default if available.
# For this example, we'll assume a successful evaluation for demonstration purposes.

# To make this runnable without an actual LLM API key, we'll mock the evaluate function's output
# In a real scenario, you'd call: result = evaluate(dataset, metrics=metrics, llm=llm, embeddings=embeddings)

# Mocking the result for demonstration purposes
# In a real scenario, these scores would be computed by Ragas using an LLM.
mock_results = {
    'faithfulness': 0.95,
    'answer_relevance': 0.92,
    'context_recall': 0.88
}

# Simulate individual scores for each example for a more realistic output
# (Ragas would return a DataFrame with scores per row and an overall average)
individual_scores = [
    {'faithfulness': 1.0, 'answer_relevance': 1.0, 'context_recall': 1.0}, # AgenticLabs.ng
    {'faithfulness': 0.9, 'answer_relevance': 0.9, 'context_recall': 0.8}, # LLMOps
    {'faithfulness': 0.9, 'answer_relevance': 0.8, 'context_recall': 0.8}, # Founders
    {'faithfulness': 0.0, 'answer_relevance': 0.1, 'context_recall': 0.0}  # Internet history (out of scope)
]

# Calculate average scores from individual scores for the overall result
overall_faithfulness = sum([s['faithfulness'] for s in individual_scores]) / len(individual_scores)
overall_answer_relevance = sum([s['answer_relevance'] for s in individual_scores]) / len(individual_scores)
overall_context_recall = sum([s['context_recall'] for s in individual_scores]) / len(individual_scores)

result = {
    'faithfulness': overall_faithfulness,
    'answer_relevance': overall_answer_relevance,
    'context_recall': overall_context_recall
}

print("\nEvaluation complete. Overall scores:")
print(result)

# --- 5. Define Pass/Fail Criteria (Thresholds) ---
# These thresholds would be configured in your CI/CD pipeline.

MIN_FAITHFULNESS = 0.85
MIN_ANSWER_RELEVANCE = 0.80
MIN_CONTEXT_RECALL = 0.75

ci_status = "PASS"

if result['faithfulness'] < MIN_FAITHFULNESS:
    print(f"FAIL: Faithfulness score ({result['faithfulness']:.2f}) is below threshold ({MIN_FAITHFULNESS:.2f}).")
    ci_status = "FAIL"

if result['answer_relevance'] < MIN_ANSWER_RELEVANCE:
    print(f"FAIL: Answer Relevance score ({result['answer_relevance']:.2f}) is below threshold ({MIN_ANSWER_RELEVANCE:.2f}).")
    ci_status = "FAIL"

if result['context_recall'] < MIN_CONTEXT_RECALL:
    print(f"FAIL: Context Recall score ({result['context_recall']:.2f}) is below threshold ({MIN_CONTEXT_RECALL:.2f}).")
    ci_status = "FAIL"

print(f"\nCI Evaluation Status: {ci_status}")

# In a real CI, a 'FAIL' status would typically block the PR merge.


### Interpreting the Code Output and Performance Trade-offs

The code above simulates a critical part of a CI-integrated evaluation pipeline. Let's break down the output and discuss its implications:

#### Interpreting the Output

1.  **RAG System Output:** You'll first see the simulated RAG system's responses for each test question, along with the `contexts` it supposedly used. This helps in debugging if the final evaluation scores are low.
2.  **Overall Scores:** The `ragas` evaluation (or its mock in our case) provides aggregate scores for key metrics:
    *   **Faithfulness:** Measures how factually consistent the generated answer is with the provided context. A high score (close to 1.0) means the answer doesn't hallucinate or introduce information not present in the context. Our example shows `0.7` because one out-of-scope question received a generic answer, which `ragas` might interpret as not being 'faithful' to a specific context, or if the mock LLM for evaluation was strict.
    *   **Answer Relevance:** Assesses if the generated answer directly addresses the question asked. A high score indicates the answer is on-topic and useful. Our example's `0.7` suggests some answers might be slightly off or too generic for the specific question.
    *   **Context Recall:** Evaluates how well the retrieved context covers all the necessary information to answer the question. A high score means the RAG system successfully retrieved relevant documents. Our example's `0.65` indicates that for some questions, the retrieved context might have been incomplete or missing critical details.
3.  **CI Evaluation Status:** Based on the predefined `MIN_` thresholds, the script determines if the overall evaluation `PASS`es or `FAIL`s. In a real CI environment, a `FAIL` status would typically prevent the Pull Request from being merged, signaling to the developer that their changes have introduced a regression in LLM performance.

#### Performance Trade-offs and Considerations

Implementing CI-integrated evaluations involves several trade-offs:

1.  **Cost:** Running LLM evaluations, especially those using powerful proprietary models (like GPT-4o, Claude 3 Opus, Gemini 1.5 Pro) for metric calculation, can incur significant API costs. Each evaluation run on a PR means multiple LLM calls. This necessitates careful selection of evaluation models (e.g., using smaller, cheaper models for some metrics, or local open-source models where feasible) and optimizing the test dataset size.
2.  **Time:** LLM inference and evaluation can be time-consuming. A comprehensive evaluation suite might take minutes or even hours to run, slowing down developer feedback loops. Strategies to mitigate this include:
    *   **Parallelization:** Running evaluations in parallel across multiple machines or containers.
    *   **Tiered Evals:** Running a fast, lightweight set of critical evaluations on every PR, and a more comprehensive, slower suite only on merges to `main` or nightly builds.
    *   **Caching:** Caching LLM responses for identical inputs if your evaluation framework supports it.
3.  **Dataset Maintenance:** The quality of your evaluations is directly tied to the quality and coverage of your test dataset. Maintaining and expanding this dataset as your application evolves is an ongoing effort. This often involves human annotation or synthetic data generation.
4.  **Metric Selection:** Choosing the right metrics is crucial. Too many metrics can increase cost and complexity; too few might miss critical issues. Focus on metrics that directly align with your application's goals (e.g., faithfulness for factual RAG, safety for chatbots, coherence for creative writing).
5.  **Flakiness:** LLM outputs can be non-deterministic, leading to flaky evaluation results. This can be addressed by:
    *   **Temperature=0:** Setting the LLM's temperature to 0 during evaluation for more deterministic outputs.
    *   **Multiple Runs:** Running evaluations multiple times and averaging results.
    *   **Robust Metrics:** Using metrics less sensitive to minor linguistic variations.

#### Typical Use Cases

*   **RAG System Updates:** Any change to the retrieval mechanism, vector database, chunking strategy, or prompt template should trigger evaluations to ensure answer quality and context adherence.
*   **Prompt Engineering Changes:** Modifications to system prompts or few-shot examples should be evaluated for desired behavior and absence of regressions.
*   **Fine-tuned Model Updates:** When deploying a new version of a fine-tuned LLM, CI evals verify that the new model performs as expected on critical tasks and doesn't introduce new biases or safety issues.
*   **Agent Tooling:** If your LLM agent gains new tools or its tool-use logic changes, evaluations can confirm correct tool invocation and response generation.
*   **Safety and Guardrails:** Automated checks for toxicity, bias, PII leakage, or adherence to safety policies on every PR.
*   **Multilingual Support:** Ensuring that changes don't negatively impact performance in different languages.

By integrating these evaluations into your CI pipeline, you establish a robust quality gate, empowering your team to build and deploy LLM applications with confidence and agility.


### Resources

*   **Ragas Documentation:** The official documentation for `ragas`, a framework for RAG evaluation. This is an excellent starting point for understanding RAG-specific metrics and implementation details.
    *   [https://docs.ragas.io/](https://docs.ragas.io/)

*   **DeepEval Documentation:** Another powerful LLM evaluation framework that supports a wide range of metrics, including custom ones, and integrates well with CI/CD.
    *   [https://docs.confident-ai.com/deepeval/](https://docs.confident-ai.com/deepeval/)

*   **GitHub Actions Documentation:** Learn how to set up custom workflows, including running Python scripts and reporting results, in your GitHub repositories.
    *   [https://docs.github.com/en/actions](https://docs.github.com/en/actions)

*   **GitLab CI/CD Documentation:** Similar to GitHub Actions, GitLab CI/CD offers robust capabilities for automating your pipelines.
    *   [https://docs.gitlab.com/ee/ci/](https://docs.gitlab.com/ee/ci/)

*   **LangChain Evaluation Module:** While `ragas` and `deepeval` are specialized, LangChain also offers an evaluation module that can be useful for basic evaluations and integrating with LangChain-based applications.
    *   [https://python.langchain.com/docs/guides/evaluation/](https://python.langchain.com/docs/guides/evaluation/)

*   **Google Gemini API Documentation:** For integrating Google's state-of-the-art LLMs into your applications and evaluations.
    *   [https://ai.google.dev/docs/gemini_api_overview](https://ai.google.dev/docs/gemini_api_overview)

*   **OpenAI API Documentation:** For integrating OpenAI's models like GPT-4o into your applications and evaluations.
    *   [https://platform.openai.com/docs/overview](https://platform.openai.com/docs/overview)
